<a href="https://colab.research.google.com/github/RatchanonPa/Data-Warehouse-and-Big-Data-Analytics/blob/main/IoT_Sleep_Stage_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [ ]:
import pandas as pd
import glob
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, LSTM, Dense, Dropout, BatchNormalization, MaxPooling1D, Input
from tensorflow.keras.callbacks import EarlyStopping, Callback
from sklearn.metrics import f1_score
from sklearn.utils import class_weight  # For class weights

In [ ]:
# prompt: แตกไฟล์ zip in drive /content/drive/MyDrive/spai-signal-sleep-staging-classification.zip

import zipfile

# Specify the path to your zip file
zip_file_path = '/content/drive/MyDrive/spai-signal-sleep-staging-classification.zip'

# Specify the directory to extract the files to
extract_dir = '/content/drive/MyDrive/extracted_files'  # You can change this

try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f"Successfully extracted files to {extract_dir}")
except FileNotFoundError:
    print(f"Error: Zip file not found at {zip_file_path}")
except zipfile.BadZipFile:
    print(f"Error: Invalid zip file at {zip_file_path}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Successfully extracted files to /content/drive/MyDrive/extracted_files


In [ ]:
# โหลดข้อมูลจากไฟล์ CSV
train_path = "/content/drive/MyDrive/extracted_files/train/train"
train_csv_files = sorted(glob.glob(f"{train_path}/*.csv"))

# อ่านข้อมูลและจัดรูปแบบใหม่
train_data = []
train_labels = []
for file in train_csv_files:
    df = pd.read_csv(file)
    # แยก features และ labels
    features = df.drop('Sleep_Stage', axis=1).values
    labels = df['Sleep_Stage'].values[::480]  # 1 label ต่อ 480 timestep

    train_data.append(features)
    train_labels.extend(labels)

# รวมข้อมูลและปรับขนาด
X_train = np.concatenate(train_data)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_train = X_train.reshape(-1, 480, X_train.shape[1])  # (samples, timesteps, features)

# เข้ารหัส labels
le = LabelEncoder()
y_train = le.fit_transform(train_labels)
y_train = tf.keras.utils.to_categorical(y_train)

In [ ]:
model = models.Sequential([
    layers.Input(shape=(480, 8)),
    layers.Conv1D(32, 3, activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(2),
    layers.LSTM(64, return_sequences=True),
    layers.LSTM(64),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(le.classes_), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ฝึกโมเดล
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=64,
    validation_split=0.2,
    callbacks=[callbacks.EarlyStopping(patience=10)]
)

Epoch 1/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 31s 27ms/step - accuracy: 0.5419 - loss: 1.2295 - val_accuracy: 0.4801 - val_loss: 1.4121
Epoch 2/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 33s 24ms/step - accuracy: 0.5802 - loss: 1.0950 - val_accuracy: 0.4540 - val_loss: 1.4279
Epoch 3/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 22s 26ms/step - accuracy: 0.5915 - loss: 1.0637 - val_accuracy: 0.4436 - val_loss: 1.4608
Epoch 4/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 21s 25ms/step - accuracy: 0.6009 - loss: 1.0209 - val_accuracy: 0.4480 - val_loss: 1.5325
Epoch 5/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 41s 24ms/step - accuracy: 0.6172 - loss: 0.9915 - val_accuracy: 0.4325 - val_loss: 1.5490
Epoch 6/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 20s 24ms/step - accuracy: 0.6283 - loss: 0.9563 - val_accuracy: 0.4364 - val_loss: 1.6504
Epoch 7/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 21s 25ms/step - accuracy: 0.6291 - loss: 0.9573 - val_accuracy: 0.4284 - val_loss: 1.6430
Epoch 8/100
835/835 ━━━━━━━━━━━━━━━━━━━━ 43s 52ms/step - accuracy: 0.6366 - loss: 0

In [ ]:
# โหลดข้อมูลทดสอบ
test_dirs = sorted(glob.glob("/content/drive/MyDrive/extracted_files/test_segment/test_segment"))

all_segments = []
all_ids = []

for dir_path in test_dirs:
    # อ่านข้อมูลและปรับขนาด
    dir_data = []
    for file in sorted(glob.glob(f"{dir_path}/*.csv")):
        df = pd.read_csv(file)
        dir_data.append(df.values)

    dir_data = np.concatenate(dir_data)
    dir_data = scaler.transform(dir_data)  # ใช้ scaler จากชุดฝึก

    # เติมข้อมูลให้ครบ 480 timestep
    if len(dir_data) % 480 != 0:
        padding = np.zeros((480 - len(dir_data)%480, 8))
        dir_data = np.vstack([dir_data, padding])

    # สร้าง ID
    num_segments = len(dir_data) // 480
    dir_ids = [f"{dir_path.split('/')[-1]}_{i:05d}" for i in range(num_segments)]

    all_segments.append(dir_data.reshape(-1, 480, 8))
    all_ids.extend(dir_ids)

# รวมข้อมูลทดสอบทั้งหมด
X_test = np.concatenate(all_segments)

ValueError: need at least one array to concatenate

In [ ]:
# ทำนายผล
predictions = model.predict(X_test)
predicted_classes = np.argmax(predictions, axis=1)
predicted_labels = le.inverse_transform(predicted_classes)

# สร้าง DataFrame และบันทึกไฟล์
submission_df = pd.DataFrame({
    'id': all_ids,
    'labels': predicted_labels
})
submission_df.to_csv('submission1.csv', index=False)

In [ ]:
print("Submission file created successfully!")
print("Unique labels in submission:", submission_df['labels'].unique())

Submission file created successfully!


NameError: name 'submission_df' is not defined